<a href="https://colab.research.google.com/github/Yoshita09/AutoCite-Intelligent-Citation-System/blob/main/AutoCite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install PyPDF2 scikit-learn
!pip install pymupdf
!pip install pdfplumber

In [ ]:
import re
import fitz
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files

# -------------------------------
# UPLOAD PDF
# -------------------------------
print("Upload PDF:")
uploaded = files.upload()
pdf_bytes = next(iter(uploaded.values()))

doc = fitz.open(stream=pdf_bytes, filetype="pdf")
text = "\n".join(page.get_text() for page in doc)
lines = [l.strip() for l in text.split("\n") if l.strip()]

# -------------------------------
# TITLE
# -------------------------------
title = "Unknown Title"
for line in lines[:40]:
    if any(x in line.lower() for x in ["doi","http","license","elsevier","issn"]):
        continue
    if 30 < len(line) < 200 and not re.search(r'\d{4}', line):
        title = line
        break

# -------------------------------
# AUTHORS (FINAL WORKING)
# -------------------------------
authors = []

# extract from first page only (important)
first_page = doc[0].get_text()

match = re.search(r"age of smart learning\s+(.*?)\s+a College", first_page, re.S)

if match:
    block = match.group(1)

    # remove commas + line breaks
    block = block.replace("\n", " ")

    authors = re.findall(r"[A-Z][a-z]+(?: [A-Z][a-z]+)+", block)

# clean
authors = list(dict.fromkeys(authors))

# -------------------------------
# YEAR
# -------------------------------
year_match = re.search(r'\b(20\d{2})\b', text)
year = year_match.group(1) if year_match else "Unknown"

# -------------------------------
# ABSTRACT (FINAL WORKING)
# -------------------------------
abstract = ""

match = re.search(r"ABSTRACT\s+(.*?)\s+1\.", text, re.S)

if match:
    abstract = match.group(1)

# clean garbage
abstract = re.sub(r"http\S+", "", abstract)
abstract = re.sub(r"creativecommons.*?license", "", abstract, flags=re.I)
abstract = re.sub(r"\s+", " ", abstract).strip()

# fallback
if len(abstract) < 50:
    abstract = " ".join(lines[15:25])

# -------------------------------
# KEYWORDS
# -------------------------------
clean_text = re.sub(r"[^a-z\s]", "", abstract.lower())

vectorizer = TfidfVectorizer(stop_words="english", max_features=15)
X = vectorizer.fit_transform([clean_text])

keywords = []
for w in vectorizer.get_feature_names_out():
    if len(w) < 5:
        continue
    if any(x in w for x in ["http", "license", "article"]):
        continue
    keywords.append(w)

keywords = keywords[:5]

# -------------------------------
#  CITATIONS [1], [2]
# -------------------------------
citations = set(re.findall(r'\[(\d+)\]', text))
citations = sorted(citations, key=int)

print(f"\n✓ Citations found: {len(citations)}")

# -------------------------------
#  REFERENCES (FORCED EXTRACTION)
# -------------------------------
references = {}

ref_section = re.search(r"References(.*)", text, re.I | re.S)

if ref_section:
    ref_lines = ref_section.group(1).split("\n")

    for line in ref_lines:
        match = re.match(r"\[(\d+)\]", line)
        if match:
            ref_id = match.group(1)
            references[ref_id] = line.strip()

print(f"✓ References extracted: {len(references)}")

# -------------------------------
# VALIDATION
# -------------------------------
valid = []
invalid = []

for c in citations:
    if c in references:
        valid.append(c)
    else:
        invalid.append(c)

# -------------------------------
# OUTPUT
# -------------------------------
print("\n📄 METADATA")
print("Title:", title)
print("Authors:", authors)
print("Year:", year)
print("Keywords:", keywords)
print("Abstract:", abstract[:400], "...")

print(f"\n✔ {len(valid)} VALID | ✖ {len(invalid)} INVALID")

print("\n✔ VALID CITATIONS (sample)")
for v in valid[:10]:
    print(f"[{v}] -> {references[v][:80]}...")

print("\n❌ INVALID CITATIONS")
print(invalid[:10])

Upload PDF:


Saving Artificial_intelligence_and_social_media_on_academic_performance_and_mental_well.pdf to Artificial_intelligence_and_social_media_on_academic_performance_and_mental_well (37).pdf

✓ Citations found: 145
✓ References extracted: 145

📄 METADATA
Title: Artificial intelligence and social media on academic performance
Authors: ['Muhammad Farrukh Shahzad', 'Shuo Xu', 'Weng Marc Lim', 'Xingbing Yang', 'Qasim Raza Khan']
Year: 2024
Keywords: ['academic', 'artificial', 'impact', 'intelligence', 'media']
Abstract: article under the CC BY license (http://creativecommons.org/licenses/by/4.0/). Research article Artificial intelligence and social media on academic performance and mental well-being: Student perceptions of positive impact in ...

✔ 145 VALID | ✖ 0 INVALID

✔ VALID CITATIONS (sample)
[1] -> [1] B. Hu, Y. Mao, K.J. Kim, How social anxiety leads to problematic use of conv...
[2] -> [2] H. Edward, et al., Artificial intelligence and obesity management : an Obesi...
[3] -> [3] P.P. R